In [198]:
import pandas as pd
import numpy as np
import sqlite3
from datasketch import MinHash, MinHashLSH
from collections import Counter
import numpy as np


In [199]:
db_path = "../DatabaseVis/main.db"
conn = sqlite3.connect(db_path)

In [200]:
query = "SELECT rxnorm_ingredient_id, meddra_id, meddra_name FROM meddra_mappings"
df = pd.read_sql_query(query, conn)

In [201]:
drug_adr_groups = df.groupby('rxnorm_ingredient_id')['meddra_id'].apply(list).to_dict()

In [202]:
id_name_dict = dict(zip(df['meddra_id'], df['meddra_name']))

In [203]:

class ADRData:
    def __init__(self, id_to_name_dict):
        """
        Initializes with a dictionary of {meddra_id: meddra_name}.
        """
        self.id_to_name = id_to_name_dict
        self.unique_ids = sorted(list(id_to_name_dict.keys()))
        
        self.id_to_idx = {adr_id: i for i, adr_id in enumerate(self.unique_ids)}
        self.idx_to_id = {i: adr_id for i, adr_id in enumerate(self.unique_ids)}
        
        self.vocab_size = len(self.unique_ids)

    def encode(self, adr_list):
        """
        Takes a list of ADR IDs and returns a binary vector (1s and 0s).
        Example: ['10028553', '10003041'] -> [0, 1, 0, 0, 1...]
        """
        vector = np.zeros(self.vocab_size, dtype=np.int8)
        
        for adr_id in adr_list:
            if adr_id in self.id_to_idx:
                idx = self.id_to_idx[adr_id]
                vector[idx] = 1
            else:
                print(f"Warning: ADR ID {adr_id} not in vocabulary.")
                
        return vector

    def decode(self, vector):
        """
        Takes a binary vector and returns a list of human-readable ADR names.
        """
        decoded_names = []
        
        active_indices = np.where(vector == 1)[0]
        
        for idx in active_indices:
            adr_id = self.idx_to_id[idx]
            name = self.id_to_name.get(adr_id, "Unknown ADR")
            decoded_names.append(name)
            
        return decoded_names


In [204]:
class ADRSignatureDB:
    def __init__(self, num_perm=128, threshold=0.5):
        self.num_perm = num_perm
        self.lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
        self.signatures = {}
        self.drug_adr_map = {} 
        self.MAX_HASH = 2**64 - 1


    def add_drug_from_vector(self, drug_id, binary_vector, original_adr_ids):

        m = MinHash(num_perm=self.num_perm)
        
        active_indices = np.where(binary_vector == 1)[0]
        
        for idx in active_indices:
            m.update(str(idx).encode('utf8'))
        
        self.lsh.insert(drug_id, m)
        self.signatures[drug_id] = m.hashvalues
        self.drug_adr_map[drug_id] = original_adr_ids

    def get_signature(self, drug_id):
        return self.signatures.get(drug_id).astype(np.uint64)

    def get_normalized_target(self, drug_id):
        raw_sig = self.get_signature(drug_id)

        if raw_sig is None:
            return None

        return (raw_sig.astype(np.float64) / self.MAX_HASH).astype(np.float32)

    def denormalize_prediction(self, predicted_floats):
        clamped = np.clip(predicted_floats, 0, 1)
        return (clamped * self.MAX_HASH).astype(np.uint64)

    def predict_vector(self, predicted_floats, vocab_size, adr_mapping_func=None):

        raw_hashvalues = self.denormalize_prediction(predicted_floats)
        
        m_query = MinHash(num_perm=self.num_perm, hashvalues=raw_hashvalues)
        neighbors = self.lsh.query(m_query)
        print(f"neighbors: {neighbors}, {m_query}")
        reconstructed_vec = np.zeros(vocab_size, dtype=np.int8)
        
        if not neighbors:
            return reconstructed_vec

        for drug_id in neighbors:
            neighbor_adr_ids = self.drug_adr_map.get(drug_id, [])
            for adr_id in neighbor_adr_ids:
                idx = adr_mapping_func(adr_id)
                if idx is not None:
                    reconstructed_vec[idx] = 1
                    
        return reconstructed_vec

In [205]:
adrMapping = ADRData(id_name_dict)
sig_db = ADRSignatureDB(num_perm=128, threshold=0.4)

In [206]:
for rxnorm_id, adr_list in drug_adr_groups.items():
    vec = adrMapping.encode(adr_list)
    
    sig_db.add_drug_from_vector(rxnorm_id, vec, adr_list)

print("ADR Signature Database is ready.")

ADR Signature Database is ready.


In [209]:
data_raw = sig_db.get_signature('1005921')
data_normalized = sig_db.get_normalized_target('1005921')


reconstructed_vec = sig_db.predict_vector(
    data_normalized, 
    vocab_size=adrMapping.vocab_size, 
    adr_mapping_func=adrMapping.id_to_idx.get
)

predicted_names = adrMapping.decode(reconstructed_vec)
print(f"Predicted ADRs: {predicted_names}")

neighbors: ['1005921', '623400'], <datasketch.minhash.MinHash object at 0x000001EA5FABA9C0>
Predicted ADRs: ['Abdominal discomfort', 'Abdominal pain', 'Abnormal sensation in eye', 'Ache', 'Acne', 'Aggression', 'Agitation', 'Agranulocytosis', 'AIDS', 'Alopecia', 'Amenorrhea', 'Angioedema', 'Anxiety', 'Asthenia', 'Ataxia', 'Atrial fibrillation', 'Atrial flutter', 'Atrioventricular block', 'AV block', 'Back pain', 'Blind', 'Blister', 'Bradycardia', 'Breast swelling', 'Breast tenderness', 'Cancer', 'Carcinogenicity', 'Chest pain', 'Chills', 'Confusion', 'Confusional state', 'Constipation', 'Convulsion', 'Coordination abnormal', 'Depression', 'Diarrhea', 'Diarrhoea', 'DIC', 'Diplopia', 'Discomfort', 'Disorientation', 'Disturbance in attention', 'Dizziness', 'Drug hypersensitivity', 'Drunkenness', 'Dry mouth', 'Dry skin', 'Dry throat', 'Dysarthria', 'Dyskinesia', 'Dysmenorrhea', 'Dyspareunia', 'Dyspepsia', 'Ectopic pregnancy', 'Emotional disorder', 'Eosinophilia', 'Epidermal necrolysis', 'Ep

In [208]:
print(data_raw)
print(data_normalized)

[ 15097273  25740402  20108236   5489579  61376268  28461822  49696357
  17194123  45286654  20625186  40617502  25207307   9514205  12848807
  13164651  53050493  15780420  23106678   6847793   9022316  42779928
  30931397   3063098 190940923  24089041   2440307  27716970  26879319
  17372404  12385893   3943068  47399880  24107112  85282569  14140767
  20136886     15015    396709 157748993  54442231  13774441  31222954
  61755685  26513921  53427104   5345927   8861935  26821375    364352
   1429394    472133  67103142   3361134  40700318   7007679  54875721
 110101212   2132121  62709776   7078582  38977085  23182271  13561678
  36849463  38110266  14071274  15775309  24962153  75164086  19214851
  27891086  87867355  28210912  58234230  45733453  62240640  63713759
  10535099   3878337  80773327  41310974   5673889  57697749  17731942
 141265943  36074241  17006578  36453924   5332004  43668018  12214400
  43318616   1046222 150828810  22880854  33508238   8394772  44940337
 10592